# Transfermarkt Web Scraping — Function Examples

This notebook presents the main scraping functions used in the project and demonstrates how each one can be executed.

The scraping logic itself is stored in the `scraping_functions.py` module. This keeps the notebook focused on documentation, testing, and examples instead of mixing the complete implementation with the analysis workflow.

## Notebook structure

1. Libraries and project setup
2. HTTP headers and league configuration
3. Scraping functions
   - `get_events()`
   - `get_matches()`
   - `get_placements()`
   - `get_squad()`
   - `get_title()`
   - `get_top_scorers()`
4. Final Notes

## 1. Libraries and Project Setup

The project uses **pandas** to organize the extracted data and a custom Python module named `scraping_functions` containing the scraping functions.

`importlib.reload()` is useful while developing the project because it reloads the module after changes are made to `scraping_functions.py`, avoiding the need to restart the Jupyter kernel.

In [2]:
import pandas as pd
import scraping_functions as sf
import importlib

importlib.reload(sf);


## 2. HTTP headers and league configuration

Additionally, a custom HTTP header containing a User-Agent string is included in every request. This header identifies the requests as originating from a standard web browser, reducing the likelihood of access restrictions and ensuring that the website returns the same content served to regular users. Using consistent headers also contributes to more reliable and stable data collection throughout the execution of the scraping functions.

In [3]:
headers = {
    "User-Agent":"Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:150.0) Gecko/20100101 Firefox/150.0"
}

## 3. Scraping Functions

Each section follows the same structure:

- purpose of the function;
- small execution example;
- conversion of the returned list into a pandas DataFrame.

This project required the development of six specialized scraping functions responsible for collecting transfermarkt data. The architecture was designed to be modular and reusable, enabling data extraction from both completed and ongoing seasons through round-specific queries. Furthermore, the same functions can be easily adapted to other competitions available on Transfermarkt, as long as they share the same underlying page structure.

The current implementation supports the following competitions:

- **Premier League (England)** → `"premier-league"`
- **Bundesliga (Germany)** → `"bundesliga"`
- **Serie A (Italy)** → `"serie-a"`
- **LaLiga (Spain)** → `"laliga"`
- **Ligue 1 (France)** → `"ligue-1"`
- **Campeonato Brasileiro Série A (Brazil)** → `"campeonato-brasileiro-serie-a"`

The examples intentionally use a limited amount of data so the notebook can be tested quickly.

### 3.1 `get_events()`

The get_events() function extracts all match events from a specific round of a Transfermarkt competition. It identifies the team involved, the minute of the event, the player responsible, and classifies the event into predefined categories, including regular goals, penalty goals, own goals, missed penalties, and red cards. The function also generates unique identifiers for each event and accounts for different page layouts to ensure accurate data extraction. The resulting data is returned in a structured format, ready for analysis or conversion into a DataFrame.

In [4]:
premier_events = []

for season in range(2024,2026):
    if season > 1994: season_round = 39
    else: season_round = 43

    for round in range(1,season_round+1):
        event_data = sf.get_events(headers,'premier-league',season,round)
        premier_events.extend(event_data)

df_events = pd.DataFrame(premier_events)
display(df_events)

,season_id,match_id,match_url,team_url,team_id,team_name,player_url,player_id,player_name,event_type,event_score,event_time_label,event_time_minute,event_time_extra
0,GB1-2024,M-2024-01-01,/spielbericht/index/spielbericht/4361261,/manchester-united/spielplan/verein/985/saison...,985,Manchester United,/joshua-zirkzee/profil/spieler/435648,435648,Joshua Zirkzee,icon-tor-formation,1:0,87',87,0
1,GB1-2024,M-2024-01-02,/spielbericht/index/spielbericht/4361262,/fc-liverpool/spielplan/verein/31/saison_id/2024,31,Liverpool FC,/diogo-jota/profil/spieler/340950,340950,Diogo Jota,icon-tor-formation,0:1,60',60,0
2,GB1-2024,M-2024-01-02,/spielbericht/index/spielbericht/4361262,/fc-liverpool/spielplan/verein/31/saison_id/2024,31,Liverpool FC,/mohamed-salah/profil/spieler/148455,148455,Mohamed Salah,icon-tor-formation,0:2,65',65,0
3,GB1-2024,M-2024-01-03,/spielbericht/index/spielbericht/4361263,/fc-arsenal/spielplan/verein/11/saison_id/2024,11,Arsenal FC,/kai-havertz/profil/spieler/309400,309400,Kai Havertz,icon-tor-formation,1:0,25',25,0
4,GB1-2024,M-2024-01-03,/spielbericht/index/spielbericht/4361263,/fc-arsenal/spielplan/verein/11/saison_id/2024,11,Arsenal FC,/bukayo-saka/profil/spieler/433177,433177,Bukayo Saka,icon-tor-formation,2:0,74',74,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2280,GB1-2025,M-2025-38-08,/spielbericht/index/spielbericht/4626175,/fc-chelsea/spielplan/verein/631/saison_id/2025,631,Chelsea FC,/wesley-fofana/profil/spieler/475411,475411,Wesley Fofana,icon-gelbrotekarte-formation,None,62',62,0
2281,GB1-2025,M-2025-38-09,/spielbericht/index/spielbericht/4626176,/tottenham-hotspur/spielplan/verein/148/saison...,148,Tottenham Hotspur,/joao-palhinha/profil/spieler/257455,257455,João Palhinha,icon-tor-formation,1:0,43',43,0
2282,GB1-2025,M-2025-38-10,/spielbericht/index/spielbericht/4626177,/west-ham-united/spielplan/verein/379/saison_i...,379,West Ham United,/taty-castellanos/profil/spieler/522784,522784,Taty Castellanos,icon-tor-formation,1:0,67',67,0
2283,GB1-2025,M-2025-38-10,/spielbericht/index/spielbericht/4626177,/west-ham-united/spielplan/verein/379/saison_i...,379,West Ham United,/jarrod-bowen/profil/spieler/314875,314875,Jarrod Bowen,icon-tor-formation,2:0,79',79,0


### 3.2 `get_match()`

The get_match() function scrapes match-level information from a specific round of a Transfermarkt competition. It collects the participating teams, final score, match date, referee, attendance, and generates unique identifiers for the season, round, and match. The function also handles different page layouts caused by the presence of forum links, ensuring that team names are extracted correctly. Finally, the collected data is organized into a structured list, making it ready for further processing or conversion into a DataFrame.

In [10]:
premier_matches = []

for season in range(2000,2002):
    if season > 1994: season_round = 38
    else: season_round = 42

    for round in range(1,season_round+1):
        matches_data = sf.get_matches(headers,'premier-league',season,round)
        premier_matches.extend(matches_data)

df_premier_matches = pd.DataFrame(premier_matches)
display(df_premier_matches)

,season_id,match_id,match_url,home_team_url,home_team_id,home_team_name,match_result,away_team_url,away_team_id,away_team_name,match_day,match_referee,match_attendance,match_time,match_time_period
0,GB1-2000,M-2000-01-01,/spielbericht/index/spielbericht/1041972,/charlton-athletic/spielplan/verein/358/saison...,358,Charlton Athletic,4:0,/manchester-city/spielplan/verein/281/saison_i...,281,Manchester City,2000-08-19,Rob Harris,20043,4:00,PM
1,GB1-2000,M-2000-01-02,/spielbericht/index/spielbericht/1041973,/fc-chelsea/spielplan/verein/631/saison_id/2000,631,Chelsea FC,4:2,/west-ham-united/spielplan/verein/379/saison_i...,379,West Ham United,2000-08-19,Graham Barber,34914,4:00,PM
2,GB1-2000,M-2000-01-03,/spielbericht/index/spielbericht/1041974,/coventry-city/spielplan/verein/990/saison_id/...,990,Coventry City,1:3,/fc-middlesbrough/spielplan/verein/641/saison_...,641,Middlesbrough FC,2000-08-19,Barry Knight,20624,4:00,PM
3,GB1-2000,M-2000-01-04,/spielbericht/index/spielbericht/1041975,/derby-county/spielplan/verein/22/saison_id/2000,22,Derby County,2:2,/fc-southampton/spielplan/verein/180/saison_id...,180,Southampton FC,2000-08-19,Andy D'Urso,27223,4:00,PM
4,GB1-2000,M-2000-01-05,/spielbericht/index/spielbericht/1041976,/leeds-united/spielplan/verein/399/saison_id/2000,399,Leeds United,2:0,/fc-everton/spielplan/verein/29/saison_id/2000,29,Everton FC,2000-08-19,Dermot Gallagher,40010,4:00,PM
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
755,GB1-2001,M-2001-38-06,/spielbericht/index/spielbericht/1023381,/leeds-united/spielplan/verein/399/saison_id/2001,399,Leeds United,1:0,/fc-middlesbrough/spielplan/verein/641/saison_...,641,Middlesbrough FC,2002-05-11,Uriah Rennie,40218,4:00,PM
756,GB1-2001,M-2001-38-07,/spielbericht/index/spielbericht/1023382,/leicester-city/spielplan/verein/1003/saison_i...,1003,Leicester City,2:1,/tottenham-hotspur/spielplan/verein/148/saison...,148,Tottenham Hotspur,2002-05-11,David Elleray,21716,4:00,PM
757,GB1-2001,M-2001-38-08,/spielbericht/index/spielbericht/1023383,/afc-sunderland/spielplan/verein/289/saison_id...,289,Sunderland AFC,1:1,/derby-county/spielplan/verein/22/saison_id/2001,22,Derby County,2002-05-11,Alan Wiley,47989,4:00,PM
758,GB1-2001,M-2001-38-09,/spielbericht/index/spielbericht/1023384,/manchester-united/spielplan/verein/985/saison...,985,Manchester United,0:0,/charlton-athletic/spielplan/verein/358/saison...,358,Charlton Athletic,2002-05-11,Graham Poll,67571,4:00,PM


### 3.3 `get_placements()`

The get_placements() function retrieves the league standings for a specific round of a Transfermarkt competition. It extracts each team's position, matches played, wins, draws, losses, goals scored, goal difference, and total points. Additionally, the function generates unique identifiers for the season and round, organizing the collected information into a structured dataset that can be easily analyzed or converted into a DataFrame.

In [11]:
premier_placements = []

for season in range(2020,2025):
    if season > 1994: season_round = 38
    else: season_round = 42

    for round in range(1,season_round+1):
        placements_data = sf.get_placements(headers,'premier-league', season, round)
        premier_placements.extend(placements_data)

df_premier_placements = pd.DataFrame(premier_placements)
display(df_premier_placements)

,season_id,team_url,team_id,team_name,position,matches_played,matches_won,matches_draw,matches_losses,matches_goals,goals_dif,points
0,GB1-2020,/fc-arsenal/spielplan/verein/11/saison_id/2020,11,Arsenal FC,1,1,1,0,0,3:0,3,3
1,GB1-2020,/leicester-city/spielplan/verein/1003/saison_i...,1003,Leicester City,2,1,1,0,0,3:0,3,3
2,GB1-2020,/fc-chelsea/spielplan/verein/631/saison_id/2020,631,Chelsea FC,3,1,1,0,0,3:1,2,3
3,GB1-2020,/manchester-city/spielplan/verein/281/saison_i...,281,Manchester City,4,1,1,0,0,2:0,2,3
4,GB1-2020,/newcastle-united/spielplan/verein/762/saison_...,762,Newcastle United,5,1,1,0,0,2:0,2,3
...,...,...,...,...,...,...,...,...,...,...,...,...
3795,GB1-2024,/wolverhampton-wanderers/spielplan/verein/543/...,543,Wolverhampton Wanderers,16,38,12,6,20,54:69,-15,42
3796,GB1-2024,/tottenham-hotspur/spielplan/verein/148/saison...,148,Tottenham Hotspur,17,38,11,5,22,64:65,-1,38
3797,GB1-2024,/leicester-city/spielplan/verein/1003/saison_i...,1003,Leicester City,18,38,6,7,25,33:80,-47,25
3798,GB1-2024,/ipswich-town/spielplan/verein/677/saison_id/2024,677,Ipswich Town,19,38,4,10,24,36:82,-46,22


### 3.4 `get_squad()`

The get_squad() function extracts squad-related information for every team participating in a Transfermarkt competition during a given season. It retrieves each team's market value, squad size, average player age, and number of foreign players. The function also handles slight variations in the page structure when extracting market values, ensuring consistent results. All collected information is organized into a structured dataset that can be easily analyzed or converted into a DataFrame.

In [13]:
premier_squad = []

# transfermarkt only contains values for team_value from 2004
for season in range(2004,2026):
    squad_data = sf.get_squad(headers,'premier-league',season)
    premier_squad.extend(squad_data)

df_premier_squad = pd.DataFrame(premier_squad)
display(df_premier_squad)

,season_id,team_url,team_id,team_name,team_squad,team_value,team_value_int,team_avg_age,team_foreigners
0,GB1-2004,/fc-chelsea/startseite/verein/631/saison_id/2004,631,Chelsea FC,31,€331.48m,331480000,24.9,24
1,GB1-2004,/manchester-united/startseite/verein/985/saiso...,985,Manchester United,37,€293.23m,293230000,24.7,25
2,GB1-2004,/fc-arsenal/startseite/verein/11/saison_id/2004,11,Arsenal FC,37,€247.00m,247000000,23.9,29
3,GB1-2004,/fc-liverpool/startseite/verein/31/saison_id/2004,31,Liverpool FC,38,€222.13m,222130000,25.3,26
4,GB1-2004,/tottenham-hotspur/startseite/verein/148/saiso...,148,Tottenham Hotspur,38,€129.45m,129449999,25.3,22
...,...,...,...,...,...,...,...,...,...
435,GB1-2025,/afc-sunderland/startseite/verein/289/saison_i...,289,Sunderland AFC,43,€446.68m,446680000,24.6,31
436,GB1-2025,/wolverhampton-wanderers/startseite/verein/543...,543,Wolverhampton Wanderers,42,€389.85m,389850000,25.0,31
437,GB1-2025,/fc-fulham/startseite/verein/931/saison_id/2025,931,Fulham FC,30,€376.20m,376200000,27.4,23
438,GB1-2025,/leeds-united/startseite/verein/399/saison_id/...,399,Leeds United,33,€374.20m,374200000,26.1,23


### 3.5 `get_title()`

The get_title() function retrieves the historical champions of a Transfermarkt competition. For each title-winning season, it extracts the champion club and its manager, while also converting the season label into a standardized season identifier. In the case of the Premier League, the function considers only seasons from 1992 onward, when the competition adopted its current format. The collected data is returned as a structured dataset, ready for analysis or conversion into a DataFrame.

In [14]:
premier_titles = sf.get_title(headers,'premier-league')

df_premier_titles = pd.DataFrame(premier_titles)
display(df_premier_titles)

,season_id,season_name,team_url,team_id,team_name,manager_url,manager_id,manager_name
0,GB1-2025,25/26,/fc-arsenal/startseite/verein/11/saison_id/2025,11,Arsenal FC,/mikel-arteta/profil/trainer/47620,47620,Mikel Arteta
1,GB1-2024,24/25,/fc-liverpool/startseite/verein/31/saison_id/2024,31,Liverpool FC,/arne-slot/profil/trainer/34822,34822,Arne Slot
2,GB1-2023,23/24,/manchester-city/startseite/verein/281/saison_...,281,Manchester City,/pep-guardiola/profil/trainer/5672,5672,Pep Guardiola
3,GB1-2022,22/23,/manchester-city/startseite/verein/281/saison_...,281,Manchester City,/pep-guardiola/profil/trainer/5672,5672,Pep Guardiola
4,GB1-2021,21/22,/manchester-city/startseite/verein/281/saison_...,281,Manchester City,/pep-guardiola/profil/trainer/5672,5672,Pep Guardiola
5,GB1-2020,20/21,/manchester-city/startseite/verein/281/saison_...,281,Manchester City,/pep-guardiola/profil/trainer/5672,5672,Pep Guardiola
6,GB1-2019,19/20,/fc-liverpool/startseite/verein/31/saison_id/2019,31,Liverpool FC,/jurgen-klopp/profil/trainer/118,118,Jürgen Klopp
7,GB1-2018,18/19,/manchester-city/startseite/verein/281/saison_...,281,Manchester City,/pep-guardiola/profil/trainer/5672,5672,Pep Guardiola
8,GB1-2017,17/18,/manchester-city/startseite/verein/281/saison_...,281,Manchester City,/pep-guardiola/profil/trainer/5672,5672,Pep Guardiola
9,GB1-2016,16/17,/fc-chelsea/startseite/verein/631/saison_id/2016,631,Chelsea FC,/antonio-conte/profil/trainer/3517,3517,Antonio Conte


### 3.6 `get_top_scorers`

The get_top_scorers() function retrieves the top scorers for a specific season of a Transfermarkt competition. Since the ranking spans multiple pages, the function first determines the total number of pages and then iterates through each one to collect all player records. For every player, it extracts their ranking position, nationality, age, club, matches played, and goals scored. The function also accounts for players who represented multiple clubs during the season, ensuring consistent data extraction. The collected information is returned as a structured dataset, ready for analysis or conversion into a DataFrame.

In [15]:
premier_top_scorers = []

for season in range(2010,2026):
    table_data = sf.get_top_scorers(headers,'premier-league',season)
    premier_top_scorers.extend(table_data)

df_premier_top_scorers = pd.DataFrame(premier_top_scorers)
display(df_premier_top_scorers)

,season_id,player_url,player_id,player_name,player_age,country_name,team_url,team_id,team_name,leaderboard_pos,matches_played,goals
0,GB1-2010,/carlos-tevez/leistungsdaten/spieler/4276/sais...,4276,Carlos Tévez,27,Argentina,/manchester-city/startseite/verein/281/saison_...,281,Manchester City,1,31,20
1,GB1-2010,/dimitar-berbatov/leistungsdaten/spieler/65/sa...,65,Dimitar Berbatov,30,Bulgaria,/manchester-united/startseite/verein/985/saiso...,985,Manchester United,2,32,20
2,GB1-2010,/robin-van-persie/leistungsdaten/spieler/4380/...,4380,Robin van Persie,27,Netherlands,/fc-arsenal/startseite/verein/11/saison_id/2010,11,Arsenal FC,3,25,18
3,GB1-2010,/darren-bent/leistungsdaten/spieler/13239/sais...,13239,Darren Bent,27,England,None,0,for 2 clubs,4,36,17
4,GB1-2010,/peter-odemwingie/leistungsdaten/spieler/12516...,12516,Peter Odemwingie,29,Nigeria,/west-bromwich-albion/startseite/verein/984/sa...,984,West Bromwich Albion,5,32,15
...,...,...,...,...,...,...,...,...,...,...,...,...
4293,GB1-2025,/matt-oriley/leistungsdaten/spieler/406634/sai...,406634,Matt O'Riley,25,Denmark,/brighton-amp-hove-albion/startseite/verein/12...,1237,Brighton & Hove Albion,276,6,1
4294,GB1-2025,/fabio-carvalho/leistungsdaten/spieler/559263/...,559263,Fábio Carvalho,23,Portugal,/fc-brentford/startseite/verein/1148/saison_id...,1148,Brentford FC,277,6,1
4295,GB1-2025,/lorenzo-lucca/leistungsdaten/spieler/572265/s...,572265,Lorenzo Lucca,25,Italy,/nottingham-forest/startseite/verein/703/saiso...,703,Nottingham Forest,278,4,1
4296,GB1-2025,/ben-davies/leistungsdaten/spieler/192765/sais...,192765,Ben Davies,32,Wales,/tottenham-hotspur/startseite/verein/148/saiso...,148,Tottenham Hotspur,279,3,1


## 4. Final Notes

Keeping the scraping functions in `scraping_functions.py` and the examples in this notebook provides a cleaner project structure:

- **`scraping_functions.py`** → function implementation;
- **this notebook** → documentation, testing, and usage examples;
- **CSV/data files** → extracted datasets.

This separation makes the project easier to maintain, debug, and expand to additional competitions.